In [8]:
import rasterio as rio
import numpy as np
import matplotlib.pyplot as plt
import os
import geopandas as gpd

In [9]:
results_dir = '/Users/willmcmahon/Documents/PhD/SCDs/results/'
index = gpd.read_file('/Users/willmcmahon/Documents/PhD/SCDs/scd-detection/data/PostIRP/index.gpkg')

covered_tiles = set()
duplicate_tiles = set()

for root, dirs, files in os.walk(results_dir):
    for file in files:
        if file.endswith('.txt'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r') as f:
                content = f.read()
                tiles = list(content.split('Tiles used: ')[1].split('\n'))[0].split(', ')
                for tile in tiles:
                    if tile in covered_tiles:
                        duplicate_tiles.add(tile)
                    else:
                        covered_tiles.add(tile)


covered_bounds = []

names = {t.split('_')[0].upper() for t in covered_tiles}
s = index.loc[index['tile_name'].isin(names), 'geometry']

duplicate_names = {t.split('_')[0].upper() for t in duplicate_tiles}
duplicates = index.loc[index['tile_name'].isin(duplicate_names), 'geometry']

In [10]:
covered = gpd.GeoDataFrame(geometry=s, crs=index.crs)
covered.to_file('/Users/willmcmahon/Documents/PhD/SCDs/scd-detection/data/PostIRP/covered_tiles.gpkg', driver='GPKG')

dups = gpd.GeoDataFrame(geometry=duplicates, crs=index.crs)
dups.to_file('/Users/willmcmahon/Documents/PhD/SCDs/scd-detection/data/PostIRP/duplicate_tiles.gpkg', driver='GPKG')